In [1]:

# Imports

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert
import helpers.test_circ_plot as circ_plot
import gc
import helpers.stc_helper as stc_helper
import time
from pycircstat2.hypothesis import rayleigh_test
import pycircstat2 as cs2
ss = hf.settings_dict()

In [2]:
baseline_tmin = 0.0
baseline_tmax = 0.3
stimulus_tmin = 0.7
stimulus_tmax = 3.7



In [3]:
from scipy.stats import ranksums
from pycircstat2.hypothesis import wheeler_watson_test, watson_u2_test

for subject_index in ss['subject_idx_list']:

    # loop over each event type
    for event_id in ss['event_id_list']:

        event_name = str(event_id)
        duty_cycle = ss['event_name_list'][event_id - 1]
        subjects_dir = ss['fs_subjects_dir']
        subject = ss['subject_id_list'][subject_index]
        print("loading dataset for subject: ", subject)

        save_dir = Path(ss['hilbert_ref_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        # load hilbert stc data
        hilbert_stc_file = Path(ss['hilbert_ref_dir']) / subject / event_name / f"{subject}-event-{event_name}-hilbert-ret-ref-vol.stc-stc.h5"
        stc = mne.read_source_estimate(hilbert_stc_file)

        stc_baseline = stc.copy().crop(baseline_tmin, baseline_tmax)
        stc_stimulus = stc.copy().crop(stimulus_tmin, stimulus_tmax)


        # stc.data shape: (n_voxels, n_times) — phase differences in radians [-pi, pi]
        n_voxels, n_times = stc.data.shape

        z_map = np.zeros(n_voxels)
        p_map = np.zeros(n_voxels)

        for voxel_idx in range(n_voxels):
            # PLV / MRL
            inst_stim = np.cos(np.imag(stc_stimulus.data[voxel_idx]))
            inst_base = np.cos(np.imag(stc_baseline.data[voxel_idx]))
            result = ranksums(inst_stim,inst_base)

            z_map[voxel_idx] =  result.statistic
            p_map[voxel_idx] =  result.pvalue


        print(f"delta_r min  : {z_map.min():.4f}")
        print(f"delta_r max  : {z_map.max():.4f}")
        print(f"delta_r mean : {z_map.mean():.4f}")
        print(f"Voxels > 0   : {(z_map > 0).sum()} / {n_voxels}")

        z_stc       = stc.copy().crop(0, 0.0 + 1/ss['fs'])
        z_stc.data  = z_map[:, np.newaxis]   # (n_voxels, 1)

        z_stc.save(save_dir / f"{subject}-event-{event_name}-z-vol.stc" , overwrite=True)


loading dataset for subject:  0005_3SJ
delta_r min  : -40.4799
delta_r max  : 34.8466
delta_r mean : 0.7818
Voxels > 0   : 797 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -40.4799
delta_r max  : 36.0840
delta_r mean : -7.4576
Voxels > 0   : 490 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -40.4799
delta_r max  : 37.7838
delta_r mean : -2.7004
Voxels > 0   : 607 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -40.3434
delta_r max  : 37.3675
delta_r mean : -7.8544
Voxels > 0   : 446 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -40.4799
delta_r max  : 37.2355
delta_r mean : -3.1537
Voxels > 0   : 6